In [29]:
import os
import pickle
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [30]:
from ucimlrepo import fetch_ucirepo

# ---------------
# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets
# ---------------

In [31]:
# •	0 → No heart disease (< 50% narrowing)
# •	1 → Mild heart disease (> 50% narrowing)
# •	2 → Moderate heart disease
# •	3 → Severe heart disease
# •	4 → Very severe heart disease

In [32]:
df = pd.concat([X, y], axis=1)
# Replace missing values marked as '?'
df.replace("?", pd.NA, inplace=True)
df = df.dropna()
# Convert all columns to numeric
df = df.apply(pd.to_numeric, errors="coerce")
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,57,0,4,140,241,0,0,123,1,0.2,2,0.0,7.0,1
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3


In [33]:
X = df.drop("num", axis=1)

# Binary classification (IMPORTANT FIX)
y = (df["num"] > 0).astype(int)

# Save feature names
feature_names = X.columns.tolist()

In [34]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


os.makedirs("model", exist_ok=True)
feature_names = X.columns.tolist()

with open("model/rf_model.pkl", "wb") as f:
    pickle.dump({"model": model, "features": feature_names}, f)
print("\nModel saved at model/rf_model.pkl")

Accuracy: 0.8666666666666667

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.91      0.88        32
           1       0.88      0.82      0.85        28

    accuracy                           0.87        60
   macro avg       0.87      0.86      0.87        60
weighted avg       0.87      0.87      0.87        60


Confusion Matrix:
 [[29  3]
 [ 5 23]]

Model saved at model/rf_model.pkl
